# T4. 프로젝트 도우미 튜터 챗봇

공개 자료실에 올라 있는 프로젝트 따라하기 노트북 13개(M1~M4 · P1~P6 · T1~T3)를 셀 단위 문서로 바꿔 Parquet 레이크에 쌓고, 바뀐 노트북만 다시 색인하는 증분 파이프라인을 만듭니다. 「M3 에서 MinIO 가 403 이 나요」 같은 질문에 프로젝트 · 단계 · 셀을 근거로 답하는 튜터를 Qdrant 와 Streamlit 으로 만들고, 질문 10개로 검색 정확도를 잽니다.

- 단계: 팀 프로젝트 · 비정형 · 예상 팀 4~5인 · 2~3주(노트북 한 번 따라하기 3~4시간)
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 데이터 엔지니어링 160H 과정 공개 자료실(github.com/aebonlee/materials, data-eng/projects) — 프로젝트 따라하기 노트북 13개(M1~M4 · P1~P6 · T1~T3), 2026-10-29 받은 스냅샷. 이 과정이 직접 만들어 공개한 노트북이라 이용허락 문제가 없음(노트북 실행 출력에 보이는 공공데이터의 출처 · 이용허락은 각 노트북 1단계에 있음)

셀을 위에서부터 차례로 실행하세요. 각 단계 끝의 **확인 포인트** 숫자가 같게 나오면 다음 단계로 갑니다.
따라하기를 한 번 끝냈다면 같은 내용의 [빈칸 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/T4/T4_blank.ipynb)으로 다시 쳐 보세요.

프로젝트를 따라하다 막히면 보통 이렇게 묻습니다. 「M3 에서 MinIO 가 403 이 나요」 「Iceberg 타임트래블은 어느 프로젝트에서 해요?」
답은 이미 노트북 안에 있습니다. 어느 프로젝트의 몇 단계, 어느 셀에 그 오류를 일부러 내 보는 코드와 설명이 있습니다. 문제는 노트북이 13개, 셀이 1,100개가 넘어 찾기 어렵다는 것입니다.

이 팀 프로젝트는 그 노트북들을 문서로 삼는 **튜터 챗봇**을 만듭니다. 답할 때는 반드시 「어느 프로젝트 · 몇 단계 · 몇 번째 셀」을 근거로 붙입니다.

```text
[수집]          자료실 목록(API) · 노트북 JSON ─▶ 셀 단위 문서(설명 · 코드 · 오류 출력) + 단계 메타
[레이크]        문서 표 Parquet(프로젝트 파티션) + 관리 표(내용 해시) ─▶ 바뀐 노트북만 다시 처리
[RAG · 서비스]  ko-sroberta 임베딩 ─▶ Qdrant ─▶ 메타 필터(프로젝트 · 단계) + 셀 종류 가중 ─▶ ask() · Streamlit
[오케스트레이션] collect ─▶ reindex ─▶ smoke(정확도 검사) 를 매일 도는 Airflow DAG
```

## 팀에서 나누는 법

8주 과정의 네 역할로 단계를 나눕니다. 단계 제목 앞의 **[역할]** 이 그 단계를 맡을 사람입니다. 노트북은 한 사람이 처음부터 끝까지 돌릴 수 있게 되어 있으니, 먼저 모두가 한 번 따라한 뒤 자기 역할의 단계를 키워 가세요.

| 역할 | 단계 | 팀에서 더 키울 것 |
| --- | --- | --- |
| 수집 | 2~3 | 새 프로젝트가 올라오면 저절로 목록에 들어오게, 셀 종류(표 · 그림 출력) 늘리기 |
| 레이크 | 4~5 | 문서 표를 Iceberg 테이블(18장)로 옮겨 색인 이력을 스냅샷으로 남기기 |
| RAG · 서비스 | 6~9 | 질문 세트를 30개로 늘려 가중값 고르기, Streamlit 화면에 「이 답이 맞았나요?」 단추 |
| 오케스트레이션 | 10 | 8주차 로컬 Airflow 에 DAG 올리기, 정확도가 떨어지면 알림 |

다 끝내면 다음 세 가지를 말할 수 있습니다.

- 노트북(JSON)을 검색할 문서로 바꿀 때 무엇을 남기고 무엇을 버리는지(실행 출력은 오류 메시지만)
- 노트북이 다시 실행되기만 하고 내용은 그대로일 때, 다시 색인하지 않으려면 무엇의 해시를 재야 하는지
- 비슷한 이름의 프로젝트(M1 · P2 둘 다 따릉이)에서 메타 필터가 없으면 엉뚱한 답이 1위로 오는 이유와, 셀 종류 가중이 정확도를 얼마나 바꾸는지

## 1단계 · 준비 — 라이브러리와 노트북 스냅샷 [모두]

임베딩은 7주차와 같은 `jhgan/ko-sroberta-multitask`(768차원), 벡터 DB 는 Qdrant 를 서버 없이(`:memory:`) 씁니다. 화면은 8주차의 Streamlit 입니다. Colab 에 없는 것만 설치합니다.

문서로 쓰는 것은 이 과정의 공개 자료실에 올라 있는 **프로젝트 따라하기 노트북 13개**입니다(M1~M4 · P1~P6 · T1~T3).
우리 과정이 만들어 공개한 노트북이라 이용허락 문제가 없습니다. 노트북 실행 출력에 보이는 공공데이터의 출처와 이용허락은 각 노트북 1단계에 적혀 있습니다.
2026-10-29 에 13개를 받아 한 파일(`t4_project_notebooks_20261029.jsonl.gz`, 한 줄 = 노트북 하나)로 묶어 두었습니다. 2단계에서는 같은 노트북을 자료실에서 직접 받아 봅니다.

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("qdrant_client", "qdrant-client")
need("sentence_transformers", "sentence-transformers")
need("streamlit")

import os, re, io, json, gzip, glob, shutil, hashlib, urllib.request
import pandas as pd
import duckdb

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

SNAPSHOT = "t4_project_notebooks_20261029.jsonl.gz"
p = fetch(SNAPSHOT)
print(p, round(os.path.getsize(p) / 1e3), "KB")

LAKE = os.environ.get("T4_LAKE", "lake")              # 레이크 뿌리 — docs/(문서 표) · manifest.parquet(관리 표)
LANDING = os.environ.get("T4_LANDING", "landing")     # 받은 노트북을 그대로 두는 곳
shutil.rmtree(LAKE, ignore_errors=True)
shutil.rmtree(LANDING, ignore_errors=True)

data/t4_project_notebooks_20261029.jsonl.gz 671 KB


> **확인 포인트** — `data/t4_project_notebooks_20261029.jsonl.gz 671 KB` 가 찍힙니다.

In [ ]:
assert round(os.path.getsize(p) / 1e3) == 671

## 2단계 · [수집] 자료실에서 노트북 받기 — 목록 API · raw 주소 · 스냅샷

자료실은 GitHub 공개 리포입니다. 두 가지 주소를 씁니다.

- **목록**: GitHub API `contents` — 폴더 안의 파일 · 폴더 이름을 JSON 으로 돌려줍니다. 키 없이 한 시간에 60번까지 부를 수 있습니다.
- **내용**: `raw.githubusercontent.com/.../<ID>/<ID>_guide.ipynb` — 노트북 파일 그대로입니다.

노트북(`.ipynb`)은 사실 JSON 파일입니다. `cells` 목록 안에 셀마다 종류(`markdown` · `code`) · 글(`source`) · 실행 출력(`outputs`)이 들어 있습니다(11장에서 API 응답 JSON 을 다룬 것과 같은 방법으로 엽니다).

In [ ]:
UA = {"User-Agent": "Mozilla/5.0 (Macintosh)"}
RAW = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects"
API = "https://api.github.com/repos/aebonlee/materials/contents/data-eng/projects"
COLAB = "https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects"
EXCLUDE = {"T4"}                                      # 튜터 자신의 노트북 — 질문 예문이 그대로 들어 있어 검색을 흐린다

def get(url, timeout=20):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout).read()

def raw_notebook(pid):
    return json.loads(get(f"{RAW}/{pid}/{pid}_guide.ipynb"))

def list_projects():
    """자료실의 프로젝트 폴더 이름(M1 · P2 · T3 …)."""
    items = json.loads(get(API))
    return sorted(x["name"] for x in items if x["type"] == "dir" and re.fullmatch(r"[MPT]\d+", x["name"]))

try:
    live_ids = list_projects()
    print("자료실의 프로젝트 폴더:", len(live_ids), "개 —", " ".join(live_ids))
except Exception as e:
    live_ids = []
    print("목록을 받지 못했습니다(스냅샷으로 계속합니다):", type(e).__name__)

자료실의 프로젝트 폴더: 19 개 — M1 M2 M3 M4 M5 M6 M7 P1 P2 P3 P4 P5 P6 P7 P8 P9 T1 T2 T3


자료실은 계속 늘어납니다. 그래서 이 숫자는 노트북을 여는 날마다 다를 수 있습니다. 확인 포인트는 고정된 스냅샷으로 겁니다.

일부러 없는 프로젝트를 받아 보세요.

In [ ]:
try:
    raw_notebook("X9")
except Exception as e:
    print("받기 실패:", type(e).__name__, getattr(e, "code", ""))

받기 실패: HTTPError 404


> **확인 포인트** — `받기 실패: HTTPError 404` — raw 주소는 없는 파일에 404 를 돌려줍니다. 목록에 없는 ID 를 손으로 넣으면 이렇게 됩니다. 그래서 수집은 목록(API)에서 출발합니다.

이제 스냅샷을 엽니다. 한 줄이 노트북 하나입니다.

In [ ]:
def load_snapshot(name=SNAPSHOT):
    with gzip.open(fetch(name), "rt", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    return {r["id"]: r["notebook"] for r in rows if r["id"] not in EXCLUDE}

notebooks = load_snapshot()
print(len(notebooks), "개:", " ".join(notebooks))
print("셀 합계:", sum(len(nb["cells"]) for nb in notebooks.values()))

13 개: M1 M2 M3 M4 P1 P2 P3 P4 P5 P6 T1 T2 T3
셀 합계: 1101


> **확인 포인트** — `13 개: M1 M2 M3 M4 P1 P2 P3 P4 P5 P6 T1 T2 T3` · `셀 합계: 1101`.

In [ ]:
assert list(notebooks) == "M1 M2 M3 M4 P1 P2 P3 P4 P5 P6 T1 T2 T3".split()
assert sum(len(nb["cells"]) for nb in notebooks.values()) == 1101

스냅샷과 자료실의 지금 노트북이 같은지 셀 목록을 견줘 봅니다. 노트북은 고쳐 다시 올리는 일이 많아, 날이 지나면 「다름」이 생길 수 있습니다. 그 「다름」을 찾아 다시 색인하는 것이 5단계 · 10단계의 일입니다.

In [ ]:
same, diff = [], []
for pid in [x for x in live_ids if x in notebooks]:
    try:
        (same if raw_notebook(pid)["cells"] == notebooks[pid]["cells"] else diff).append(pid)
    except Exception as e:
        print(pid, "받기 실패:", type(e).__name__)
print("같음:", len(same), "· 다름:", diff, "· 스냅샷 뒤에 새로 생긴 폴더:", [x for x in live_ids if x not in notebooks and x not in EXCLUDE])

같음: 13 · 다름: [] · 스냅샷 뒤에 새로 생긴 폴더: ['M5', 'M6', 'M7', 'P7', 'P8', 'P9']


## 3단계 · [수집] 셀을 문서로 — 단계 메타 붙이고, 출력은 오류만

셀 하나를 문서 하나로 삼습니다. 문서마다 **어느 프로젝트 · 몇 단계 · 몇 번째 셀 · 어떤 종류**인지를 메타로 붙입니다. 나중에 답의 근거로 보여 주고, 검색 범위를 좁히는(필터) 데 씁니다.

- 단계: 설명 셀이 `## N단계 · 제목` 으로 시작하면 그다음 셀부터 N단계입니다. 1단계 앞은 0단계(들어가는 글)로 둡니다.
- 종류: 설명(`md`) · 코드(`code`) · 오류 출력(`error`) 셋입니다.
- **실행 출력은 오류 메시지만** 문서로 씁니다. 표 · 숫자 출력은 그 노트북의 데이터일 뿐 질문에 대한 답이 아닙니다. 반면 「실패: HTTP 403 Forbidden」 같은 줄은 수강생이 화면에서 보고 그대로 물어보는 글이라 가장 쓸모 있는 문서입니다.
  오류 줄만으로는 무엇을 하다 난 오류인지 모르므로, 그 셀 코드의 앞부분 200자를 함께 붙입니다.

In [ ]:
STAGE = re.compile(r"^## (\d+)단계 · (.+)")
ERR = re.compile(r"(실패|오류|막힘|막혔|거절|Error|Exception|HTTP \d{3}|Forbidden|refused)")

def out_text(c):
    parts = []
    for o in c.get("outputs", []):
        if o["output_type"] == "stream":
            parts.append("".join(o["text"]))
        elif "text/plain" in o.get("data", {}):
            parts.append("".join(o["data"]["text/plain"]))
    return "\n".join(parts)

def parse_notebook(pid, nb):
    cells = nb["cells"]
    title = "".join(cells[0]["source"]).splitlines()[0].split(". ", 1)[1]      # 「# M3. 지하철 혼잡 시간표」
    stage, stage_title, docs = 0, "들어가는 글", []
    for i, c in enumerate(cells):
        src = "".join(c["source"]).strip()
        if not src:
            continue
        m = STAGE.match(src) if c["cell_type"] == "markdown" else None
        if m:
            stage, stage_title = int(m.group(1)), m.group(2).strip()
            continue
        base = {"project": pid, "title": title, "stage": stage, "stage_title": stage_title, "cell": i}
        if c["cell_type"] == "code":
            docs.append(base | {"kind": "code", "text": src})
            errs = [line for line in out_text(c).splitlines() if ERR.search(line)]
            if errs:
                docs.append(base | {"kind": "error", "text": "\n".join(errs)[:500] + "\n" + src[:200]})
        else:
            docs.append(base | {"kind": "md", "text": src})
    return docs

docs = [d for pid, nb in notebooks.items() for d in parse_notebook(pid, nb)]
doc_df = pd.DataFrame(docs)
print(len(doc_df), doc_df.groupby("kind").size().to_dict())
doc_df[(doc_df["project"] == "M3") & (doc_df["kind"] == "error")][["stage", "stage_title", "text"]]

1027 {'code': 426, 'error': 46, 'md': 555}


,stage,stage_title,text
110,2,읽기 — 일부러 기본 키 검사에 걸리기,"실패: 기본 키가 겹칩니다\nKEY = [""사용월"", ""호선명"", ""지하철역""]\n..."
120,3,넓은 표를 긴 표로,실패: invalid literal for int() with base 10: '0...
131,4,Parquet 로 저장 — 월별 폴더,실패: BinderException — Binder Error: Referenced...
145,6,DuckDB 로 s3:// 바로 조회 — 시간대별 붐비는 역,실패: HTTP 403 Forbidden\ncon = duckdb.connect()...


> **확인 포인트** — 문서 1,027개(설명 555 · 코드 426 · 오류 46). M3 의 오류 문서 가운데 **6단계 「DuckDB 로 s3:// 바로 조회」의 `실패: HTTP 403 Forbidden`** 이 보입니다 — 질문 「M3 에서 MinIO 가 403 이 나요」의 답이 있는 자리입니다.

In [ ]:
assert len(doc_df) == 1027 and doc_df.groupby("kind").size().to_dict() == {"code": 426, "error": 46, "md": 555}
m3_403 = doc_df[(doc_df["project"] == "M3") & doc_df["text"].str.startswith("실패: HTTP 403")]
assert m3_403["stage"].tolist() == [6]

> **Aebon's Tip** — 단계 표시는 원고 형식(`## N단계 · 제목`)에 기대고 있습니다. 같은 형식으로 만든 노트북이면 새 프로젝트가 늘어도 코드를 고칠 필요가 없습니다. 형식이 다른 노트북을 넣을 때는 「단계 0 이 너무 많은 노트북」을 찾아 보세요 — 단계를 못 읽었다는 뜻입니다.

일부러 깨진 노트북을 열어 보세요. 받다가 끊긴 파일은 JSON 끝이 잘려 있습니다.

In [ ]:
broken = json.dumps(notebooks["M1"])[:5000]
try:
    parse_notebook("M1", json.loads(broken))
except json.JSONDecodeError as e:
    print("열기 실패:", type(e).__name__, "—", str(e)[:60])

열기 실패: JSONDecodeError — Unterminated string starting at: line 1 column 4979 (char 49


> **자주 겪는 오류** — 받기가 끊기면 이렇게 열리지 않으므로 오히려 다행입니다. 더 위험한 것은 열리기는 하는데 내용이 옛것인 경우입니다 — 그것은 5단계의 내용 해시로 잡습니다.

## 4단계 · [레이크] 조각내기 — 임베딩 모델의 입력 한도

임베딩 모델은 글이 길면 **앞부분만 읽고 나머지는 조용히 버립니다**. ko-sroberta 의 한도를 먼저 확인합니다(7주차 청킹).

In [ ]:
from sentence_transformers import SentenceTransformer

_cache = {}
def model():
    if "emb" not in _cache:
        _cache["emb"] = SentenceTransformer("jhgan/ko-sroberta-multitask")
    return _cache["emb"]

emb = model()
print("입력 한도:", emb.max_seq_length, "토큰")
longest = doc_df.loc[doc_df["text"].str.len().idxmax()]
n_tok = len(emb.tokenizer(longest["text"])["input_ids"])
print("가장 긴 셀:", longest["project"], longest["stage"], "단계 ·", len(longest["text"]), "자 ·", n_tok, "토큰")

입력 한도: 128 토큰
가장 긴 셀: T2 10 단계 · 5933 자 · 3163 토큰


일부러 긴 셀을 통째로 넣어 보세요. 앞 128토큰만 남긴 글과 임베딩이 사실상 같습니다 — 셀 뒤쪽 내용은 검색에 전혀 쓰이지 않는다는 뜻입니다.

In [ ]:
ids = emb.tokenizer(longest["text"], add_special_tokens=False)["input_ids"][:100]
head_only = emb.tokenizer.decode(ids)
v_full, v_head = emb.encode([longest["text"], head_only], normalize_embeddings=True)
print("통째 vs 앞부분만 — 코사인:", round(float(v_full @ v_head), 3))

통째 vs 앞부분만 — 코사인: 0.994


그래서 셀을 줄 단위로 모아 300자 안쪽 조각으로 자릅니다. 한 줄이 300자보다 길면 그 줄을 다시 자릅니다.
조각마다 「프로젝트 이름 · 단계 제목」을 앞에 붙여 임베딩합니다. 「s3:// 조회」 단계의 코드 조각이 「지하철 혼잡 시간표」의 것이라는 맥락이 벡터에 들어갑니다.

In [ ]:
def chunk(text, limit=300):
    out, buf = [], ""
    for line in text.splitlines():
        while len(line) > limit:
            if buf:
                out.append(buf)
                buf = ""
            out.append(line[:limit])
            line = line[limit:]
        if buf and len(buf) + len(line) + 1 > limit:
            out.append(buf)
            buf = line
        else:
            buf = buf + "\n" + line if buf else line
    if buf.strip():
        out.append(buf)
    return out

def to_chunks(docs):
    return [d | {"part": k, "text": piece} for d in docs for k, piece in enumerate(chunk(d["text"]))]

def embed_text(r):
    return f"{r['title']} · {r['stage_title']}\n{r['text']}"

chunks = pd.DataFrame(to_chunks(docs))
print(len(chunks), "조각 · 가장 긴 조각", chunks["text"].str.len().max(), "자")
duckdb.sql("SELECT project, count(*) AS n, count(DISTINCT stage) AS stages FROM chunks GROUP BY project ORDER BY project").df().T

1544 조각 · 가장 긴 조각 300 자


,0,1,2,3,4,5,6,7,8,9,10,11,12
project,M1,M2,M3,M4,P1,P2,P3,P4,P5,P6,T1,T2,T3
n,46,79,96,103,131,118,121,98,119,111,178,164,180
stages,8,9,9,10,10,10,10,11,10,11,11,12,12


> **확인 포인트** — 가장 긴 셀은 수천 자 · 수천 토큰인데 모델은 128토큰만 읽고, 통째 · 앞부분만 두 임베딩의 코사인이 1에 가깝습니다. 조각은 **1,544개**, 가장 긴 조각도 300자입니다.

In [ ]:
assert emb.max_seq_length == 128 and n_tok > 1000 and float(v_full @ v_head) > 0.99
assert len(chunks) == 1544 and chunks["text"].str.len().max() == 300

## 5단계 · [레이크] 문서 표 Parquet 과 증분 — 바뀐 노트북만 다시

임베딩은 이 파이프라인에서 가장 오래 걸리는 일입니다(조각 1,500개에 1~3분). 노트북 하나가 바뀔 때마다 13개를 모두 다시 임베딩하면 낭비입니다.
그래서 레이크에 두 표를 둡니다.

- **문서 표** `lake/docs/project=<ID>/part-0.parquet` — 조각 · 메타 · 벡터. 프로젝트마다 폴더를 나눕니다(Hive 파티션, 4주차 17장). 바뀐 프로젝트는 그 폴더만 갈아 끼웁니다.
- **관리 표** `lake/manifest.parquet` — 프로젝트마다 내용 해시 · 조각 수 · 처리 시각.

새로 받은 노트북의 해시가 관리 표와 같으면 건너뜁니다(6주차 체크포인트와 같은 생각입니다 — 어디까지 처리했는지를 남겨 둡니다).
**무엇의 해시를 재느냐**가 중요합니다. 파일 전체가 아니라 3단계에서 뽑은 **문서**(검색에 들어가는 글)의 해시를 잽니다. 이유는 바로 다음 실패에서 봅니다.

In [ ]:
def doc_hash(docs):
    blob = json.dumps(docs, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()

def refresh(notebooks, lake=LAKE):
    """바뀐 노트북만 다시 조각 · 임베딩해 문서 표의 그 파티션을 갈아 끼웁니다. 처리한 프로젝트 ID 목록을 돌려줍니다."""
    from datetime import datetime, timezone
    man_path = f"{lake}/manifest.parquet"
    man = pd.read_parquet(man_path) if os.path.exists(man_path) else pd.DataFrame(columns=["project", "sha", "chunks", "at"])
    old = {r.project: r for r in man.itertuples()}
    done, rows = [], []
    for pid, nb in sorted(notebooks.items()):
        if pid in EXCLUDE:
            continue
        docs = parse_notebook(pid, nb)
        h = doc_hash(docs)
        if pid in old and old[pid].sha == h:
            rows.append(old[pid]._asdict())
            continue
        part = pd.DataFrame(to_chunks(docs))
        part["vec"] = list(model().encode([embed_text(r) for r in part.to_dict("records")], batch_size=64,
                                          normalize_embeddings=True, show_progress_bar=False))
        folder = f"{lake}/docs/project={pid}"
        shutil.rmtree(folder, ignore_errors=True)
        os.makedirs(folder)
        part.drop(columns="project").to_parquet(f"{folder}/part-0.parquet", index=False)   # project 는 폴더 이름에
        rows.append({"project": pid, "sha": h, "chunks": len(part), "at": datetime.now(timezone.utc).isoformat()})
        done.append(pid)
    for pid in set(old) - set(notebooks):                  # 자료실에서 사라진 프로젝트는 레이크에서도 뺀다
        shutil.rmtree(f"{lake}/docs/project={pid}", ignore_errors=True)
        done.append(pid)
    os.makedirs(lake, exist_ok=True)
    pd.DataFrame(rows, columns=["project", "sha", "chunks", "at"]).to_parquet(man_path, index=False)
    if done:
        _cache.pop("qd", None)                            # 색인이 낡았으니 다음 검색 때 다시 만든다
    return done

처음 한 번은 13개를 모두 처리합니다(Colab CPU 에서 2~3분).

In [ ]:
import time
t0 = time.time()
first = refresh(notebooks)
print("처리:", len(first), "개 ·", round(time.time() - t0), "초")
again = refresh(notebooks)
print("다시 돌리면:", again)
lake_n = duckdb.sql(f"SELECT project, count(*) AS n FROM read_parquet('{LAKE}/docs/*/*.parquet', hive_partitioning=true) GROUP BY project ORDER BY project").df()
print(lake_n.set_index("project")["n"].to_dict())

처리: 13 개 · 72 초


다시 돌리면: []
{'M1': 46, 'M2': 79, 'M3': 96, 'M4': 103, 'P1': 131, 'P2': 118, 'P3': 121, 'P4': 98, 'P5': 119, 'P6': 111, 'T1': 178, 'T2': 164, 'T3': 180}


> **확인 포인트** — 처음에는 13개를 처리하고, 바로 다시 돌리면 `[]` — 하나도 다시 처리하지 않습니다. 레이크의 조각 수 합계는 4단계와 같은 1,544개, M3 은 96조각입니다.

In [ ]:
assert len(first) == 13 and again == []
assert int(lake_n["n"].sum()) == 1544 and lake_n.set_index("project")["n"]["M3"] == 96

일부러 **파일 해시**로 바뀜을 재 보세요. 노트북을 다시 실행하기만 하면 셀마다 실행 번호(`execution_count`)가 새로 붙습니다. 내용은 그대로입니다.

In [ ]:
import copy
rerun = copy.deepcopy(notebooks["M3"])
for k, c in enumerate(x for x in rerun["cells"] if x["cell_type"] == "code"):
    c["execution_count"] = k + 1                         # 다시 실행한 노트북 흉내

file_hash = lambda nb: hashlib.sha256(json.dumps(nb).encode()).hexdigest()
print("파일 해시 같은가:", file_hash(rerun) == file_hash(notebooks["M3"]))
print("문서 해시 같은가:", doc_hash(parse_notebook("M3", rerun)) == doc_hash(parse_notebook("M3", notebooks["M3"])))
print("refresh:", refresh(notebooks | {"M3": rerun}))

파일 해시 같은가: False
문서 해시 같은가: True
refresh: []


> **확인 포인트** — 파일 해시는 `False`(바뀐 것처럼 보임), 문서 해시는 `True`, refresh 는 `[]` 입니다. 파일 해시로 판단했다면 노트북이 다시 실행될 때마다 헛되이 다시 임베딩했을 것입니다.

In [ ]:
assert file_hash(rerun) != file_hash(notebooks["M3"]) and refresh(notebooks | {"M3": rerun}) == []

이번에는 진짜로 내용이 바뀐 경우입니다. M3 에 설명 셀을 하나 덧붙인 판(연습용으로 고친 것)을 넣으면 M3 만 다시 처리합니다.

In [ ]:
edited = copy.deepcopy(notebooks["M3"])
edited["cells"].append({"cell_type": "markdown", "metadata": {}, "source": ["(연습용으로 덧붙인 셀) MinIO 콘솔은 9001 포트입니다."]})
print("고친 판:", refresh(notebooks | {"M3": edited}))
print("되돌리기:", refresh(notebooks))
print(pd.read_parquet(f"{LAKE}/manifest.parquet")[["project", "chunks"]].T)

고친 판: ['M3']


되돌리기: ['M3']
         0   1   2    3    4    5    6   7    8    9    10   11   12
project  M1  M2  M3   M4   P1   P2   P3  P4   P5   P6   T1   T2   T3
chunks   46  79  96  103  131  118  121  98  119  111  178  164  180


> **확인 포인트** — `고친 판: ['M3']` · `되돌리기: ['M3']` — 다른 12개는 건드리지 않습니다.

In [ ]:
assert refresh(notebooks | {"M3": edited}) == ["M3"] and refresh(notebooks) == ["M3"]
assert refresh(notebooks) == []

## 6단계 · [RAG] Qdrant 에 싣고 메타 필터로 찾기

레이크의 문서 표를 읽어 Qdrant 컬렉션 하나에 싣습니다. 벡터는 이미 레이크에 있으므로 다시 임베딩하지 않습니다 — 색인을 날려도 몇 초면 다시 만듭니다.
메타(프로젝트 · 단계 · 종류)는 Qdrant 의 **payload** 로 함께 넣습니다. 검색할 때 `Filter` 로 「M3 안에서만」 「6단계 안에서만」 「코드 셀만」처럼 범위를 좁힙니다(7주차 메타데이터 필터).

In [ ]:
from qdrant_client import QdrantClient, models

COLLECTION = "t4_cells"

def build_index(lake=LAKE):
    from qdrant_client import QdrantClient, models
    df = duckdb.sql(f"SELECT * FROM read_parquet('{lake}/docs/*/*.parquet', hive_partitioning=true)").df()
    qd = QdrantClient(":memory:")
    qd.create_collection(COLLECTION, vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE))
    meta = df.drop(columns="vec").to_dict("records")
    qd.upsert(COLLECTION, points=[models.PointStruct(id=i, vector=[float(x) for x in v], payload=m)
                                  for i, (v, m) in enumerate(zip(df["vec"], meta))])
    return qd

def index(lake=LAKE):
    if "qd" not in _cache:
        _cache["qd"] = build_index(lake)
    return _cache["qd"]

def search(qd, q, project=None, stage=None, kinds=None, k=100):
    from qdrant_client import models
    must = []
    if project:
        must.append(models.FieldCondition(key="project", match=models.MatchValue(value=project)))
    if stage is not None:
        must.append(models.FieldCondition(key="stage", match=models.MatchValue(value=stage)))
    if kinds:
        must.append(models.FieldCondition(key="kind", match=models.MatchAny(any=kinds)))
    vec = model().encode(q, normalize_embeddings=True).tolist()
    return qd.query_points(COLLECTION, query=vec, query_filter=models.Filter(must=must) if must else None,
                           limit=k, with_payload=True).points

qd = index()
print("실은 조각:", qd.count(COLLECTION).count)
q = "MinIO 가 403 이 나요"
for label, hits in [("필터 없이", search(qd, q, k=3)), ("M3 안에서", search(qd, q, project="M3", k=3))]:
    print(label)
    for h in hits:
        print(f"   {h.payload['project']} {h.payload['stage']}단계 셀 {h.payload['cell']:3} {h.payload['kind']:5} {h.score:.3f}  {h.payload['text'][:40]!r}")

실은 조각: 1544


필터 없이
   P5 3단계 셀  25 md    0.466  '> **확인 포인트** — `press-raw: 83 개 | 37.4 M'
   M3 5단계 셀  41 code  0.450  'from minio import Minio\nfrom minio.error'
   T1 5단계 셀  43 code  0.448  'from minio import Minio\nfrom minio.error'
M3 안에서
   M3 5단계 셀  41 code  0.450  'from minio import Minio\nfrom minio.error'
   M3 5단계 셀  41 code  0.442  '    print("실패:", e.code)'
   M3 5단계 셀  38 code  0.439  '    osname = {"Linux": "linux", "Darwin"'


> **확인 포인트** — 1,544조각이 실립니다. 필터 없이 찾으면 1위가 MinIO 버킷을 다루는 **P5 3단계**입니다 — 「MinIO」라는 말은 여러 프로젝트에 나오기 때문입니다. `project="M3"` 로 좁히면 M3 안에서 찾습니다.

In [ ]:
assert qd.count(COLLECTION).count == 1544
assert search(qd, q, k=1)[0].payload["project"] == "P5"
assert {h.payload["project"] for h in search(qd, q, project="M3", k=20)} == {"M3"}

그런데 M3 안에서도 1위는 아직 5단계(MinIO 띄우기)입니다. 403 이 실제로 난 곳은 6단계인데, 「403」이라는 숫자는 임베딩에서 의미가 약합니다. 이것은 7단계에서 고칩니다.

일부러 단계 번호를 **글자**로 넘겨 보세요. payload 의 `stage` 는 정수라서 `"6"` 과 같은 값이 하나도 없습니다. 오류가 나지 않고 **빈 결과**가 옵니다 — 조용한 실패라 더 위험합니다.

In [ ]:
try:
    top = search(qd, q, project="M3", stage="6", k=1)[0]
except IndexError:
    print("검색 결과 0건 — stage 는 정수로 넘겨야 합니다")
print("정수로:", search(qd, q, project="M3", stage=6, k=1)[0].payload["text"][:30])

검색 결과 0건 — stage 는 정수로 넘겨야 합니다


정수로: Trino 가 하던 일 — 객체 저장소의 Parquet


## 7단계 · [RAG] 질문 읽기 · 셀 종류 가중 · 낱말 일치 보너스

튜터의 검색은 세 겹입니다.

1. **질문에서 메타 읽기** — 질문에 `M3` · `P2` 같은 프로젝트 ID 나 「6단계」가 있으면 필터로 씁니다. 한글 바로 앞뒤에 붙어도(「M3에서」) 잡히게 영문 · 숫자 경계만 봅니다.
2. **셀 종류 가중** — 「~가 나요」「실패」「403」처럼 **오류를 묻는 질문**이면 오류 출력 조각을 조금 올리고(×1.1), 「어디서 해요?」「방법」처럼 **찾아가는 질문**이면 설명 셀을 올리고 코드 · 오류는 내립니다(설명 ×1.05 · 코드 ×0.95 · 오류 ×0.9).
3. **낱말 일치 보너스** — 질문의 낱말(조사를 떼고)이 조각 · 단계 제목에 그대로 들어 있으면 낱말마다 +0.05 를 줍니다. 「403」「타임트래블」처럼 임베딩이 약한 숫자 · 고유 낱말을 살립니다(7주차 BM25 하이브리드의 아주 단순한 판).

Qdrant 에서 후보 100개를 받아, 그 안에서 점수를 다시 매겨(재순위) 1위 조각의 프로젝트 · 단계를 답으로 삼습니다.

In [ ]:
ERRQ = re.compile(r"(오류|에러|실패|안 ?돼|Error|Exception|\b\d{3}\b)", re.I)
W_ERR = {"md": 1.0, "code": 1.0, "error": 1.1}
W_HOW = {"md": 1.05, "code": 0.95, "error": 0.9}
KW = 0.05
STOP = {"프로젝트", "어느", "어디", "단계", "방법", "해요", "나요", "돼요", "에서", "오류", "실패", "있나요"}
JOSA = re.compile(r"(은|는|이|가|을|를|에서|으로|로|의|도|만|와|과|하는|해요|나요|돼요|이요)$")

def parse_question(q):
    pm = re.search(r"(?<![A-Za-z0-9])([MPT]\d{1,2})(?![0-9])", q)
    sm = re.search(r"(\d{1,2})\s*단계", q)
    return (pm.group(1) if pm else None), (int(sm.group(1)) if sm else None)

def terms(q):
    out = set()
    for t in re.findall(r"[A-Za-z0-9_\-\.]{2,}|[가-힣]{2,}", q):
        t = JOSA.sub("", t) if re.match("[가-힣]", t) else t.lower()
        if len(t) >= 2 and t not in STOP and not re.fullmatch(r"[mpt]\d{1,2}", t):
            out.add(t)
    return out

def rerank(q, hits, weight=True, kw=KW):
    w = (W_ERR if ERRQ.search(q) else W_HOW) if weight else {"md": 1, "code": 1, "error": 1}
    ts = terms(q)
    scored = []
    for h in hits:
        p = h.payload
        hay = f"{p['title']} {p['stage_title']} {p['text']}".lower()
        bonus = sum(t in hay for t in ts)
        scored.append((h.score * w[p["kind"]] + kw * bonus, p))
    return sorted(scored, key=lambda x: -x[0])

def tutor(qd, q, use_filter=True, weight=True, kw=KW, kinds=None):
    project, stage = parse_question(q) if use_filter else (None, None)
    hits = search(qd, q, project=project, stage=stage, kinds=kinds, k=100)
    return rerank(q, hits, weight=weight, kw=kw)

print(parse_question("M3에서 MinIO 가 403 이 나요"), parse_question("P1 7단계 타임트래블"), parse_question("따릉이 대여소"))
print(sorted(terms("Iceberg 타임트래블은 어느 프로젝트에서 해요?")))
for s, p in tutor(qd, "M3 에서 MinIO 가 403 이 나요")[:3]:
    print(f"{s:.3f} {p['project']} {p['stage']}단계 셀 {p['cell']} {p['kind']:5} {p['text'][:40]!r}")

('M3', None) ('P1', 7) (None, None)
['iceberg', '타임트래블']


0.539 M3 6단계 셀 48 error '실패: HTTP 403 Forbidden\ncon = duckdb.conn'
0.509 M3 5단계 셀 41 code  'from minio import Minio\nfrom minio.error'
0.496 M3 5단계 셀 37 md    'MinIO 는 S3 와 같은 말(API)을 쓰는 객체 저장소입니다(4주차'


> **확인 포인트** — `('M3', None) ('P1', 7) (None, None)` · 낱말은 `['iceberg', '타임트래블']`. 이제 1위가 **M3 6단계의 오류 조각 `실패: HTTP 403 Forbidden`** 입니다.

In [ ]:
assert parse_question("M3에서 MinIO 가 403 이 나요") == ("M3", None) and parse_question("P1 7단계 타임트래블") == ("P1", 7)
assert sorted(terms("Iceberg 타임트래블은 어느 프로젝트에서 해요?")) == ["iceberg", "타임트래블"]
best = tutor(qd, "M3 에서 MinIO 가 403 이 나요")[0][1]
assert (best["project"], best["stage"], best["kind"]) == ("M3", 6, "error")

## 8단계 · [RAG] 질문 10개로 정확도 재기 — 그리고 일부러 빼 보기

「잘 되는 것 같다」로는 가중값을 고를 수 없습니다. 기대 답(프로젝트 · 단계)을 정해 둔 질문 세트로 **1위가 맞는 비율**을 잽니다.
질문은 수강생이 실제로 할 법한 말투로, 오류형 · 찾아가기형을 섞었습니다. 기대 답은 노트북을 열어 확인한 자리입니다.

In [ ]:
QUESTIONS = [
    ("M3 에서 MinIO 가 403 이 나요", "M3", 6),
    ("Iceberg 타임트래블은 어느 프로젝트에서 해요?", "P1", 7),
    ("CSV 를 읽는데 utf-8 codec can't decode 오류가 나요", "M1", 2),
    ("Debezium 커넥터 등록이 500 으로 실패해요", "T3", 4),
    ("MongoDB 에 같은 리뷰를 다시 넣으면 duplicate key 오류가 나요", "P6", 3),
    ("법령 조문을 항 · 호 단위로 자르는 방법", "P4", 4),
    ("생활인구를 넣다가 외래 키 오류가 나요", "P3", 5),
    ("M1 에서 대여소 번호 앞자리 0 이 사라져요", "M1", 2),
    ("Airflow DAG 순서를 어기면 build_gold 가 실패해요", "T2", 10),
    ("Spark 로 날씨와 따릉이를 조인하는 단계", "P2", 6),
]

def evaluate(qd, questions=QUESTIONS, **opt):
    rows = []
    for q, pid, stage in questions:
        top = tutor(qd, q, **opt)[0][1]
        rows.append({"question": q, "want": f"{pid} {stage}", "got": f"{top['project']} {top['stage']}",
                     "kind": top["kind"], "ok": (top["project"], top["stage"]) == (pid, stage)})
    res = pd.DataFrame(rows)
    return res["ok"].mean(), res

acc, res = evaluate(qd)
print("정확도:", acc)
res

정확도: 0.9


,question,want,got,kind,ok
0,M3 에서 MinIO 가 403 이 나요,M3 6,M3 6,error,True
1,Iceberg 타임트래블은 어느 프로젝트에서 해요?,P1 7,P6 8,md,False
2,CSV 를 읽는데 utf-8 codec can't decode 오류가 나요,M1 2,M1 2,error,True
3,Debezium 커넥터 등록이 500 으로 실패해요,T3 4,T3 4,error,True
4,MongoDB 에 같은 리뷰를 다시 넣으면 duplicate key 오류가 나요,P6 3,P6 3,error,True
5,법령 조문을 항 · 호 단위로 자르는 방법,P4 4,P4 4,md,True
6,생활인구를 넣다가 외래 키 오류가 나요,P3 5,P3 5,md,True
7,M1 에서 대여소 번호 앞자리 0 이 사라져요,M1 2,M1 2,md,True
8,Airflow DAG 순서를 어기면 build_gold 가 실패해요,T2 10,T2 10,md,True
9,Spark 로 날씨와 따릉이를 조인하는 단계,P2 6,P2 6,md,True


> **확인 포인트** — 10개 중 **9개**가 맞습니다(0.9). 틀린 하나는 「Iceberg 타임트래블은 어느 프로젝트에서 해요?」로, P1 7단계 대신 **P6 8단계**(Iceberg 테이블에 쌓기 — 「시간 여행」 Tip 이 있는 곳)가 1위입니다.
> 사실 P6 8단계에도 `scan(snapshot_id=…)` 시간 여행이 나오므로 완전히 틀린 답은 아닙니다. 「기대 답이 하나뿐인가」도 질문 세트를 만들 때 팀이 정할 일입니다.

In [ ]:
assert acc == 0.9 and res[~res["ok"]]["want"].tolist() == ["P1 7"] and res[~res["ok"]]["got"].tolist() == ["P6 8"]

이제 일부러 하나씩 빼 봅니다. 같은 질문 10개, 같은 색인입니다.

In [ ]:
trials = {
    "전체(필터 + 가중 + 낱말)": {},
    "낱말 보너스 없이": {"kw": 0},
    "셀 종류 가중 없이": {"weight": False},
    "메타 필터 없이": {"use_filter": False},
    "코드 셀만": {"kinds": ["code"]},
    "설명 셀만": {"kinds": ["md"]},
}
board = {}
for name, opt in trials.items():
    a, r = evaluate(qd, **opt)
    board[name] = a
    print(f"{name:18} {a:.1f}  틀림: {r[~r['ok']]['want'].tolist()}")

전체(필터 + 가중 + 낱말)   0.9  틀림: ['P1 7']


낱말 보너스 없이          0.6  틀림: ['M3 6', 'P1 7', 'M1 2', 'M1 2']


셀 종류 가중 없이         0.8  틀림: ['M3 6', 'P1 7']


메타 필터 없이           0.8  틀림: ['P1 7', 'M1 2']


코드 셀만              0.6  틀림: ['M3 6', 'P1 7', 'M1 2', 'M1 2']


설명 셀만              0.8  틀림: ['M3 6', 'P1 7']


> **확인 포인트** — 전체 0.9 · 낱말 보너스 없이 0.6 · 가중 없이 0.8 · 메타 필터 없이 0.8 · **코드 셀만 0.6** · **설명 셀만 0.8**.
> 코드 셀만 넣으면 답이 설명 셀에 있는 질문(앞자리 0 · utf-8 읽기 실패)과 오류 출력에 있는 질문(403)을 모두 놓칩니다. 설명 셀만 넣으면 오류 출력이 빠져 「403」 질문을 놓칩니다.
> 가중을 빼도 「403」을 놓칩니다 — 오류형 질문에서 오류 조각을 올려 준 것이 그 한 문제를 맞힌 이유입니다. 셀 종류를 모두 넣고 질문에 따라 가중하는 쪽이 낫습니다.

In [ ]:
assert board == {"전체(필터 + 가중 + 낱말)": 0.9, "낱말 보너스 없이": 0.6, "셀 종류 가중 없이": 0.8,
                 "메타 필터 없이": 0.8, "코드 셀만": 0.6, "설명 셀만": 0.8}

메타 필터가 없을 때 틀리는 질문을 자세히 봅니다. M1 과 P2 는 **둘 다 따릉이 데이터**를 쓰고, 둘 다 「대여소 번호 앞자리 0」을 설명합니다(P2 는 「M1 과 같은 이유」라고까지 적어 두었습니다).

In [ ]:
q = "M1 에서 대여소 번호 앞자리 0 이 사라져요"
for label, opt in [("필터 없이", {"use_filter": False}), ("필터(M1)", {})]:
    s, p = tutor(qd, q, **opt)[0]
    print(f"{label:6} → {p['project']} {p['stage']}단계 「{p['stage_title']}」 {s:.3f}  {p['text'][:50]!r}")

필터 없이  → P2 2단계 「따릉이 — 날짜 × 대여소 표 읽기」 0.723  '대여소 번호는 글자로 읽습니다. 숫자로 읽으면 `00102` 의 앞자리 0이 사라져 `10'


필터(M1) → M1 2단계 「읽기 — 일부러 한 번 실패하기」 0.589  '`dtype=str` 로 모든 열을 글자로 읽었습니다. 대여소 번호 `00502` 의 앞자'


> **확인 포인트** — 필터 없이는 **P2 2단계**(따릉이 — 날짜 × 대여소 표 읽기)가 1위, 질문의 `M1` 을 필터로 쓰면 **M1 2단계**(읽기 — 일부러 한 번 실패하기)입니다. 질문에 적힌 「M1」이라는 두 글자는 임베딩에서는 거의 힘이 없습니다 — 그래서 메타로 따로 읽어 필터로 씁니다.

In [ ]:
assert tutor(qd, q, use_filter=False)[0][1]["project"] == "P2" and tutor(qd, q)[0][1]["project"] == "M1"

> **Aebon's Tip** — 질문 세트 10개는 출발점일 뿐입니다. RAG 담당은 프로젝트마다 2~3개씩 30개로 늘리고, 가중값(1.1 · 1.05 · 0.05)을 바꿔 가며 표를 다시 그려 보세요. 질문 세트에 맞춰 값을 너무 정밀하게 고르면 새 질문에서 오히려 떨어집니다 — 질문을 반으로 나눠 한쪽으로 고르고 다른 쪽으로 재는 것이 안전합니다.

## 9단계 · [서비스] ask() 와 Streamlit 화면

튜터의 입구는 함수 하나 `ask(질문)` 입니다. 답에는 늘 근거를 붙입니다 — 프로젝트 · 단계 · 셀 번호와, 그 노트북을 Colab 에서 바로 여는 주소입니다.
LLM(7주차 `gpt-4o-mini`)은 키가 있을 때만 근거 조각으로 문장을 다듬습니다. 키가 없으면 「어디를 보면 되는지」와 근거 조각을 그대로 보여 줍니다.

In [ ]:
def ask(q, qd=None, k=3):
    qd = qd or index()
    ranked = tutor(qd, q)
    top = ranked[0][1]
    seen, cites = set(), []
    for s, p in ranked:
        key = (p["project"], p["stage"], p["cell"])
        if key not in seen:
            seen.add(key)
            cites.append({"project": p["project"], "stage": p["stage"], "cell": p["cell"], "kind": p["kind"],
                          "score": round(s, 3), "text": p["text"]})
        if len(cites) == k:
            break
    text = (f"{top['project']} 「{top['title']}」 {top['stage']}단계 「{top['stage_title']}」를 보세요. "
            f"{top['cell']}번 셀({top['kind']}) 근처입니다.")
    return {"project": top["project"], "stage": top["stage"], "text": text, "cites": cites,
            "colab": f"{COLAB}/{top['project']}/{top['project']}_guide.ipynb"}

def openai_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.environ.get("OPENAI_API_KEY")

def llm(result, q):
    """키가 있을 때만 — 근거 조각만으로 답을 다듬습니다. 없으면 None."""
    key = openai_key()
    if not key:
        return None
    from openai import OpenAI
    ctx = "\n\n".join(f"[{c['project']} {c['stage']}단계 셀 {c['cell']} · {c['kind']}]\n{c['text']}" for c in result["cites"])
    r = OpenAI(api_key=key).chat.completions.create(model="gpt-4o-mini", temperature=0, messages=[
        {"role": "system", "content": "데이터 엔지니어링 과정의 튜터입니다. 근거 조각에 있는 내용만으로 짧게 답하고, 어느 프로젝트 몇 단계를 보면 되는지 꼭 적으세요."},
        {"role": "user", "content": ctx + "\n\n질문: " + q}])
    return r.choices[0].message.content

r = ask("M3 에서 MinIO 가 403 이 나요")
print(r["text"])
print(r["colab"])
for c in r["cites"]:
    print(" 근거:", c["project"], c["stage"], "단계 셀", c["cell"], c["kind"], c["score"], repr(c["text"][:40]))
print(llm(r, "M3 에서 MinIO 가 403 이 나요") or "(OPENAI_API_KEY 가 없어 근거 조각만 보여 줍니다)")

M3 「지하철 혼잡 시간표」 6단계 「DuckDB 로 s3:// 바로 조회 — 시간대별 붐비는 역」를 보세요. 48번 셀(error) 근처입니다.
https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/M3/M3_guide.ipynb
 근거: M3 6 단계 셀 48 error 0.539 '실패: HTTP 403 Forbidden\ncon = duckdb.conn'
 근거: M3 5 단계 셀 41 code 0.509 'from minio import Minio\nfrom minio.error'
 근거: M3 5 단계 셀 37 md 0.496 'MinIO 는 S3 와 같은 말(API)을 쓰는 객체 저장소입니다(4주차'
(OPENAI_API_KEY 가 없어 근거 조각만 보여 줍니다)


> **확인 포인트** — `M3 「지하철 혼잡 시간표」 6단계 「DuckDB 로 s3:// 바로 조회 — 시간대별 붐비는 역」를 보세요.` 로 시작하고, 근거 1위가 6단계의 오류 조각입니다.

In [ ]:
assert r["text"].startswith("M3 「지하철 혼잡 시간표」 6단계") and r["cites"][0]["kind"] == "error"
assert r["colab"].endswith("/M3/M3_guide.ipynb")

Streamlit 화면과 10단계의 Airflow 는 노트북 밖의 프로세스에서 돕니다. 그래서 지금까지 만든 함수를 **모듈 파일 하나**(`t4_core.py`)로 옮깁니다. 손으로 다시 쓰지 않고 `inspect.getsource` 로 노트북의 함수를 그대로 옮기므로, 노트북을 고치면 모듈도 같이 바뀝니다.

In [ ]:
import inspect

CONSTS = ["DATA", "SNAPSHOT", "UA", "RAW", "API", "COLAB", "EXCLUDE", "COLLECTION", "STAGE", "ERR",
          "ERRQ", "W_ERR", "W_HOW", "KW", "STOP", "JOSA", "QUESTIONS"]
FUNCS = [fetch, get, raw_notebook, list_projects, load_snapshot, out_text, parse_notebook, model, chunk, to_chunks,
         embed_text, doc_hash, refresh, build_index, index, search, parse_question, terms, rerank, tutor, evaluate,
         ask, openai_key, llm]

def write_module(path="t4_core.py", extra=()):
    head = ('"""T4 프로젝트 도우미 튜터 — 노트북 2~10단계의 함수를 옮긴 모듈(노트북이 만든 파일, 손으로 고치지 말 것)."""\n'
            "import os, re, io, json, gzip, glob, shutil, hashlib, urllib.request\nimport pandas as pd\nimport duckdb\n\n"
            '_cache = {}\nLAKE = os.environ.get("T4_LAKE", "lake")\nLANDING = os.environ.get("T4_LANDING", "landing")\n')
    consts = "\n".join(f"{k} = {globals()[k]!r}" for k in CONSTS)
    body = "\n\n".join(inspect.getsource(f) for f in [*FUNCS, *extra])
    open(path, "w", encoding="utf-8").write(head + consts + "\n\n\n" + body)

write_module()
import importlib, t4_core
importlib.reload(t4_core)
t4_core._cache["emb"] = model()                          # 같은 프로세스 안이면 모델을 다시 읽지 않게
print([q for q, *_ in QUESTIONS[:3]])
print("모듈의 답:", [t4_core.ask(q)["text"][:12] for q, *_ in QUESTIONS[:3]])

['M3 에서 MinIO 가 403 이 나요', 'Iceberg 타임트래블은 어느 프로젝트에서 해요?', "CSV 를 읽는데 utf-8 codec can't decode 오류가 나요"]


모듈의 답: ['M3 「지하철 혼잡 시', 'P6 「리뷰 감성 파이', 'M1 「서울 따릉이 대']


> **확인 포인트** — 모듈의 `ask` 가 노트북의 `ask` 와 질문 10개 모두 같은 답을 냅니다.

In [ ]:
assert [t4_core.ask(q)["text"] for q, *_ in QUESTIONS] == [ask(q)["text"] for q, *_ in QUESTIONS]

화면은 8주차의 채팅 화면(`st.chat_input` · `st.chat_message`)입니다. 색인 만들기는 무거우므로 `st.cache_resource` 로 한 번만 합니다.

In [ ]:
%%writefile app.py
import streamlit as st
import t4_core

st.set_page_config(page_title="프로젝트 도우미 튜터")
st.title("프로젝트 도우미 튜터")
st.caption("프로젝트 노트북 13개에서 찾아 답합니다. 프로젝트 ID(M3 · P2 …)를 함께 쓰면 더 정확합니다. 예: M3 에서 MinIO 가 403 이 나요")

@st.cache_resource
def ready():
    return t4_core.index()

qd = ready()
if "history" not in st.session_state:
    st.session_state.history = []
for role, text in st.session_state.history:
    st.chat_message(role).write(text)

q = st.chat_input("어디서 막혔나요?")
if q:
    st.chat_message("user").write(q)
    r = t4_core.ask(q, qd)
    text = t4_core.llm(r, q) or r["text"]
    with st.chat_message("assistant"):
        st.write(text)
        st.markdown(f"[노트북 열기(Colab)]({r['colab']})")
        for c in r["cites"]:
            with st.expander(f"근거 — {c['project']} {c['stage']}단계 · 셀 {c['cell']} · {c['kind']}"):
                st.code(c["text"])
        st.caption("근거: " + ", ".join(f"{c['project']} {c['stage']}단계" for c in r["cites"]))
    st.session_state.history += [("user", q), ("assistant", text)]

Writing app.py


내 컴퓨터에서 화면을 열 때는 터미널에서 아래처럼 실행하고 브라우저로 http://localhost:8501 에 들어갑니다(Colab 에서는 화면을 바로 열 수 없어 아래 `AppTest` 로 확인합니다).

```bash
streamlit run app.py
```

In [ ]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(os.path.abspath("app.py"), default_timeout=300).run()
at.chat_input[0].set_value("Iceberg 타임트래블 P1 7단계 어떻게 해요?").run()
said = [m.markdown[0].value for m in at.chat_message if m.markdown]
print("화면 제목:", at.title[0].value)
print("화면의 말풍선:", said)
print(at.caption[-1].value)

화면 제목: 프로젝트 도우미 튜터
화면의 말풍선: ['Iceberg 타임트래블 P1 7단계 어떻게 해요?', 'P1 「아파트 실거래가 대시보드」 7단계 「타임트래블 — 「어제 기준」으로 다시 보기」를 보세요. 77번 셀(md) 근처입니다.']
근거: P1 7단계, P1 7단계, P1 7단계


> **확인 포인트** — 제목 `프로젝트 도우미 튜터`, 답이 `P1 「아파트 실거래가 대시보드」 7단계` 로 시작하고, 아래에 `근거: P1 7단계, …` 가 붙습니다. 예외는 0건입니다.

In [ ]:
assert not at.exception and at.title[0].value == "프로젝트 도우미 튜터"
assert any(s.startswith("P1 「아파트 실거래가 대시보드」 7단계") for s in said) and at.caption[-1].value.startswith("근거: P1 7단계")

## 10단계 · [오케스트레이션] Airflow DAG — 노트북이 바뀌면 다시 색인

자료실의 노트북은 고쳐 다시 올라옵니다. 매일 한 번 세 태스크를 돌립니다.

| 태스크 | 하는 일 | 남기는 것 |
| --- | --- | --- |
| collect | 목록 API → 노트북 받기(받지 못하면 스냅샷) | landing/*.ipynb |
| reindex | landing → 내용 해시가 바뀐 노트북만 다시 조각 · 임베딩 | lake/docs · lake/manifest.parquet |
| smoke | 질문 10개로 정확도 재기 — 0.8 아래면 실패 | (실패하면 재시도 뒤 멈춤) |

태스크 사이에서 넘기는 것은 **파일**(landing · lake)뿐입니다. 메모리의 Qdrant(`:memory:`)는 다음 태스크로 넘어가지 않으므로, smoke 는 레이크에서 색인을 다시 만듭니다(벡터가 레이크에 있어 몇 초면 됩니다). Streamlit 화면도 같은 레이크를 읽습니다.

In [ ]:
def collect(source=None, landing=LANDING):
    """노트북을 landing 에 받아 둡니다. source: live(자료실) | snapshot. 돌려주는 값은 받은 프로젝트 ID 목록."""
    source = source or os.environ.get("T4_SOURCE", "live")
    nbs = {}
    if source == "live":
        try:
            nbs = {pid: raw_notebook(pid) for pid in list_projects() if pid not in EXCLUDE}
        except Exception as e:
            print("자료실에서 받지 못해 스냅샷을 씁니다:", type(e).__name__)
    if not nbs:
        nbs = load_snapshot()
    shutil.rmtree(landing, ignore_errors=True)
    os.makedirs(landing)
    for pid, nb in nbs.items():
        json.dump(nb, open(f"{landing}/{pid}.ipynb", "w", encoding="utf-8"), ensure_ascii=False)
    return sorted(nbs)

def reindex(landing=LANDING, lake=LAKE):
    nbs = {os.path.basename(p)[:-6]: json.load(open(p, encoding="utf-8")) for p in sorted(glob.glob(f"{landing}/*.ipynb"))}
    return refresh(nbs, lake)

def smoke(lake=LAKE, need=0.8):
    _cache.pop("qd", None)
    acc, res = evaluate(index(lake))
    if acc < need:
        raise ValueError(f"튜터 정확도 {acc:.1f} < {need} — 틀린 질문: {res[~res['ok']]['question'].tolist()}")
    return acc

write_module(extra=[collect, reindex, smoke])
print(open("t4_core.py", encoding="utf-8").read().count("\ndef "), "개 함수가 t4_core.py 에")

27 개 함수가 t4_core.py 에


In [ ]:
os.makedirs("dags", exist_ok=True)

In [ ]:
%%writefile dags/t4_tutor_reindex.py
from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.python import PythonOperator

import t4_core as T

with DAG(
    dag_id="t4_tutor_reindex",
    start_date=datetime(2026, 11, 1),
    schedule="@daily",
    catchup=False,
    default_args={"owner": "t4", "retries": 2, "retry_delay": timedelta(minutes=5)},
    tags=["data-eng", "T4"],
) as dag:
    collect = PythonOperator(task_id="collect", python_callable=T.collect)
    reindex = PythonOperator(task_id="reindex", python_callable=T.reindex)
    smoke = PythonOperator(task_id="smoke", python_callable=T.smoke)

    collect >> reindex >> smoke

Writing dags/t4_tutor_reindex.py


이 노트북(Colab)에는 Airflow 가 없습니다. 일부러 DAG 파일을 불러 보면 이렇게 실패합니다. 대신 파일이 파이썬 문법에 맞는지와 태스크 이름 · 순서를 `ast` 로 읽어 확인합니다.

In [ ]:
import ast, sys
sys.path.insert(0, "dags")
try:
    import t4_tutor_reindex
except ModuleNotFoundError as e:
    print("불러오기 실패:", e)

tree = ast.parse(open("dags/t4_tutor_reindex.py", encoding="utf-8").read())
task_ids = [kw.value.value for node in ast.walk(tree) if isinstance(node, ast.Call)
            for kw in node.keywords if kw.arg == "task_id"]
print("태스크:", task_ids)

불러오기 실패: No module named 'airflow'
태스크: ['collect', 'reindex', 'smoke']


> **확인 포인트** — `No module named 'airflow'` 로 불러오기가 실패하고, 태스크는 `['collect', 'reindex', 'smoke']` 입니다.

In [ ]:
assert task_ids == ["collect", "reindex", "smoke"]

DAG 가 하루에 한 번 할 일을 노트북에서 함수 호출로 한 번 돌립니다. 이 노트북의 확인 포인트가 날마다 바뀌지 않도록 여기서는 스냅샷으로 받습니다(`T4_SOURCE=snapshot`). Airflow 에서는 기본값 `live` 로 자료실에서 받습니다.

In [ ]:
os.environ["T4_SOURCE"] = "snapshot"
importlib.reload(t4_core)
t4_core._cache["emb"] = model()
for task in [t4_core.collect, t4_core.reindex, t4_core.smoke]:
    print("완료:", task.__name__, task())

완료: collect ['M1', 'M2', 'M3', 'M4', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'T1', 'T2', 'T3']


완료: reindex []


완료: smoke 0.9


> **확인 포인트** — collect 가 13개를 받고, reindex 는 `[]`(5단계에서 이미 색인했고 내용이 같음), smoke 는 `0.9` 입니다. 한 번 더 돌려도 다시 임베딩하지 않습니다 — 매일 돌아도 바뀐 노트북이 없으면 몇 초면 끝납니다.

In [ ]:
assert len(t4_core.collect()) == 13 and t4_core.reindex() == [] and t4_core.smoke() == 0.9

smoke 가 정말로 나쁜 색인을 막는지, 일부러 **코드 셀만 남은** 레이크를 만들어 재 보세요(수집 쪽에서 셀 종류를 잘못 거른 상황입니다).

In [ ]:
bad = "lake_bad"
shutil.rmtree(bad, ignore_errors=True)
shutil.copytree(LAKE, bad)
for f in glob.glob(f"{bad}/docs/*/*.parquet"):
    d = pd.read_parquet(f)
    d[d["kind"] == "code"].to_parquet(f, index=False)
msg = ""
try:
    t4_core.smoke(lake=bad)
except ValueError as e:
    msg = str(e)
    print("smoke 실패:", msg[:70])
_ = t4_core._cache.pop("qd", None)                   # 다음 검색은 진짜 레이크로

smoke 실패: 튜터 정확도 0.6 < 0.8 — 틀린 질문: ['M3 에서 MinIO 가 403 이 나요', 'Iceberg 타임트래블은 어


> **확인 포인트** — `smoke 실패: 튜터 정확도 0.6 < 0.8 — …` — Airflow 에서는 이 태스크가 재시도 2번 뒤 실패로 멈추므로, 색인이 나빠진 것을 사람이 바로 알아챕니다.

In [ ]:
assert msg.startswith("튜터 정확도 0.6 < 0.8")

> **Aebon's Tip** — 8주차 로컬 Airflow 로 옮길 때는 `dags/` 에 `t4_tutor_reindex.py` 와 `t4_core.py` 를 함께 넣고, `_PIP_ADDITIONAL_REQUIREMENTS` 에 `qdrant-client sentence-transformers` 를 넣으세요. `T4_LAKE` · `T4_LANDING` 을 컨테이너 안의 경로로 맞추면 됩니다.
> 지금은 smoke 가 실패해도 reindex 가 이미 레이크를 바꿔 놓은 뒤입니다. 새 색인을 옆 폴더(`lake_next`)에 만들고 smoke 를 통과하면 그때 바꿔 끼우는 방식(blue-green)이 오케스트레이션 담당의 다음 과제입니다.

## 11단계 · 더 해 보기 · 발표 준비

- **수집**: 자료실에는 이 스냅샷 뒤로도 프로젝트가 늘고 있습니다. `T4_SOURCE=live` 로 collect · reindex 를 돌려 새 프로젝트가 저절로 들어오는지 보세요. 실행 출력의 표(`text/plain`) 가운데 열 이름만 뽑아 문서로 넣으면 「`rent_station` 열은 어디서 만들어요?」 같은 질문에도 답할 수 있습니다.
- **레이크**: 문서 표를 Iceberg 테이블(18장)로 옮기세요. reindex 한 번이 스냅샷 하나가 되어 「지난주 색인으로 답하면 어땠나」를 타임트래블로 다시 볼 수 있습니다(P1 7단계).
- **RAG · 서비스**: 질문 세트를 30개로 늘리고, 질문을 반으로 나눠 가중값을 고르세요. Streamlit 화면에 「이 답이 맞았나요?」 단추를 달아 답과 근거를 파일로 남기면 다음 질문 세트가 저절로 쌓입니다.
- **오케스트레이션**: 8주차 로컬 Airflow 에 DAG 를 올리고, smoke 가 실패하면 알림이 가게 하세요. 위 Tip 의 blue-green 바꿔 끼우기도 해 보세요.
- 발표에서는 「질문 세트가 10개뿐이라는 점」 「기대 답을 하나로 정한 질문(타임트래블)」 「튜터 자신의 노트북은 문서에서 뺀 이유」를 한계와 판단으로 함께 밝히세요.